# 🎬 PeakCut Studio: 1-Click Autonomous Web Dashboard (Colab Edition)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shreeharsh-patil/PeakCut/blob/main/PeakCut_Colab.ipynb)

**Zero External API Keys • Local Gemma on NVIDIA GPU • Multi-Worker Parallel Rendering • Auto-Saved to Google Drive • Full Web Dashboard**

Run the full **PeakCut Studio Website Dashboard** with free NVIDIA GPU acceleration. Everything — from pasting YouTube/Twitch/Kick links, uploading local videos, multi-worker parallel clip rendering, previewing, to scheduling YouTube Shorts — is handled **directly from the web dashboard**, and all generated clips are **saved automatically in your Google Drive** (`MyDrive/PeakCut_Shorts`).

### ⚡ Quick Start:
1. Select GPU Runtime: **Runtime** → **Change runtime type** → select **T4 GPU**.
2. *(Optional)* Add `YOUTUBE_CLIENT_ID`, `YOUTUBE_CLIENT_SECRET`, `YOUTUBE_REFRESH_TOKEN`, or `YOUTUBE_COOKIES` in Colab **Secrets** (🔑 icon in the left sidebar) — they are auto-loaded on startup.
3. Click the **Run button (▶)** on the cell below.
4. Click the generated **`https://*.trycloudflare.com`** link to launch PeakCut Studio in your browser!

In [ ]:
# @title 🚀 Launch PeakCut Studio (1-Click Start)
# @markdown Mounts Google Drive, configures free NVIDIA GPU acceleration, loads Google's open **Gemma** model locally,
# @markdown spins up parallel clip & job workers, and launches the **PeakCut Studio Website Dashboard** with a free public Cloudflare link.
GEMMA_MODEL = "gemma2:2b" # @param ["gemma2:2b", "gemma2:9b", "paligemma"]
WHISPER_MODEL = "base" # @param ["tiny", "base", "small", "medium", "large-v3"]
CLIP_WORKERS = 4 # @param {type:"slider", min:2, max:8, step:1}
MAX_CONCURRENT_JOBS = 4 # @param {type:"slider", min:1, max:6, step:1}
SAVE_TO_GOOGLE_DRIVE = True # @param {type:"boolean"}
RUN_IN_BACKGROUND = False # @param {type:"boolean"}

import importlib, os, sys, subprocess

# 1. Install system tools & packages
print("📦 Step 1/4: Installing system dependencies (ffmpeg, fonts, zstd)...")
subprocess.run(
    "apt-get update -qq && apt-get install -y -qq ffmpeg libsm6 libxext6 libgl1 fonts-freefont-ttf fonts-dejavu-core zstd curl",
    shell=True, check=True
)

# 2. Clone or sync latest PeakCut repository
REPO_URL = "https://github.com/shreeharsh-patil/PeakCut.git"
REPO_DIR = "/content/PeakCut"

print(f"📥 Step 2/4: Syncing latest PeakCut repository from {REPO_URL}...")
if not os.path.exists(REPO_DIR):
    subprocess.run(f"git clone {REPO_URL} {REPO_DIR}", shell=True, check=True)
else:
    subprocess.run(f"git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} reset --hard origin/main", shell=True, check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 3. Install Python dependencies & latest yt-dlp
print("🐍 Step 3/4: Installing Python dependencies & updating yt-dlp...")
subprocess.run("pip install -q -r requirements.txt && pip install -q -U yt-dlp", shell=True)

# Optional: Auto-load YouTube OAuth / cookies from Colab Secrets if configured
try:
    from google.colab import userdata
    loaded_secrets = []
    for secret_key in ("YOUTUBE_CLIENT_ID", "YOUTUBE_CLIENT_SECRET", "YOUTUBE_REFRESH_TOKEN", "YOUTUBE_ACCESS_TOKEN", "YOUTUBE_COOKIES"):
        try:
            val = userdata.get(secret_key)
            if val and val.strip():
                os.environ[secret_key] = val.strip()
                loaded_secrets.append(secret_key)
        except Exception:
            pass
    if loaded_secrets:
        print(f"🔑 Loaded Colab Secrets: {', '.join(loaded_secrets)}")
except Exception:
    pass

# 4. Setup local GPU environment, parallel workers & launch PeakCut Studio
print("🚀 Step 4/4: Initializing local Gemma GPU model, multi-worker pool & PeakCut Studio...")
import colab_runner
importlib.reload(colab_runner)

colab_runner.setup_colab_environment(
    model_name=GEMMA_MODEL,
    mount_drive=SAVE_TO_GOOGLE_DRIVE,
    whisper_model=WHISPER_MODEL
)

# Apply selected parallel worker settings
os.environ["CLIP_WORKERS"] = str(CLIP_WORKERS)
os.environ["MAX_CONCURRENT_JOBS"] = str(MAX_CONCURRENT_JOBS)
os.environ["NUM_QUEUE_WORKERS"] = str(MAX_CONCURRENT_JOBS)
os.environ["LLM_WORKERS"] = "3"
os.environ["OLLAMA_NUM_PARALLEL"] = "4"
os.environ["FFMPEG_ENCODER"] = "auto"
print(f"⚡ Parallel Workers Active: {MAX_CONCURRENT_JOBS} job workers × {CLIP_WORKERS} clip render workers + 3 AI workers")

colab_runner.launch_web_ui(run_in_background=RUN_IN_BACKGROUND)


In [ ]:
# @title 📋 Server Status, Active Jobs & Live Logs (Optional)
# Run this cell to check active server processes, job states, or inspect recent logs
import json, urllib.request
print("🔍 Active Server Processes:")
!ps aux | grep -E "uvicorn|cloudflared|ollama" | grep -v grep || echo "No running server found."

print("\n🎬 Live Jobs Snapshot:")
try:
    with urllib.request.urlopen("http://127.0.0.1:8000/api/jobs", timeout=5) as r:
        jobs = json.loads(r.read().decode())
        for j in jobs[:5]:
            print(f"  • [{j.get('status')}] {j.get('title')} — {j.get('stage')} ({int((j.get('progress') or 0)*100)}%) | clips={len(j.get('clips') or [])}")
            if j.get("error"):
                print(f"      ⚠️ Error: {j['error']}")
        if not jobs:
            print("  (No jobs yet)")
except Exception as e:
    print(f"  Server not reachable yet: {e}")

print("\n📜 Recent Server Logs:")
!tail -n 30 /content/uvicorn.log 2>/dev/null || echo "(Running in foreground mode in cell above)"
